In [2]:
import os
import sys
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import gseapy as gp
import anndata as ad
import statistics
import tempfile
import sklearn
import cosg
import leidenalg
import celltypist
import muon as mu
from tqdm import tqdm
sc._settings.n_jobs= 24
sc.settings.verbosity = 1
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [3]:
def check_dict_duplicates(dict):
    seen = set()
    dup = any(item in seen or seen.add(item) for lst in dict.values() for item in lst)
    if dup==False:
        return '无重复'
    else:
        return '有重复'

In [4]:
obj_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R5/'

In [5]:
finnal_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/'

In [6]:
leiden_groups=['L4_leiden_TOTALVI_0.1','L4_leiden_TOTALVI_0.5', 'L4_leiden_TOTALVI_1', 'L4_leiden_TOTALVI_1.5', 'L4_leiden_TOTALVI_0.3','L4_leiden_TOTALVI_0.8']

In [7]:
def get_annotation_data(celltype):
    adata = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_count_scRNA.h5ad",backed='r')
    adt = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_preprocess_scADT.h5ad")
    adata.obsm = adt.obsm
    
    # read Level1, tcr, bcr infor
    indices = pd.read_csv(f'{obj_path}/{celltype}/R5_indices_{celltype}.csv',index_col=0)
    # join leiden groups
    leiden_data = adt.obs.loc[:,leiden_groups]
    adata.obs = adata.obs.join(indices, how='left')
    adt.obs = adt.obs.join(indices, how='left')
    adata.obs = adata.obs.join(leiden_data, how='left')
    adata.strings_to_categoricals()
    return adata,adt,leiden_data

In [8]:
def get_norm_annotation_data(celltype):
    adata = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_count_scRNA.h5ad")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adt = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_preprocess_scADT.h5ad")
    adata.obsm = adt.obsm
    
    # read Level1, tcr, bcr infor
    indices = pd.read_csv(f'{obj_path}/{celltype}/R5_indices_{celltype}.csv',index_col=0)
    # join leiden groups
    leiden_data = adt.obs.loc[:,leiden_groups]
    adata.obs = adata.obs.join(indices, how='left')
    adt.obs = adt.obs.join(indices, how='left')
    adata.obs = adata.obs.join(leiden_data, how='left')
    return adata,adt,leiden_data

# 1. 定 CD4 Treg Cell Refine

In [104]:
celltype="TregCD4"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [105]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [106]:
groupby = "L4_leiden_TOTALVI_1"

In [274]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [276]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='var',groupby=groupby)

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False,layer='denoised_protein')
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='IKZF2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='IL2RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
sc.pl.dotplot(adata, ['ISG15','CD160','KLRD1','IL2RA','CTLA4','FOXP3','FCRL3','CCR10','RORC','KLRB1','HLA-DRB1',
                     'HAVCR2','LAG3','IKZF1','IKZF2','IKZF3','TIGIT','ENTPD3','IL7R','TNFRSF18','BMI1','BCL6','CD44'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['CD40LG','CCR4','CCR6','CXCR5','KLRB1','GZMK','CCL5','BTBD9','CXCR3',
                      'GATA3','CCR10','LIMS1','RORC','CD27','CCR1','CCR2','PRDM1',"FAS"],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.violin(adata,["IKZF2",'IKZF3','FOXP3'],groupby='L4_leiden_TOTALVI_1.5',size=0)

In [ ]:
## sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_1.5']=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [107]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1
3     83793
0     76515
6     75720
4     71784
8     48377
7     29943
2     29708
1     29656
5     22200
9      3680
10     1431
Name: count, dtype: int64

In [108]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#IRF4- TGFB1+ CXCR4lo
cell_dict = {
            'Treg Naïve CD4+ T':['2','3',],#CD45RA+ FOXP3+
    'Treg memory CD4+ T':['1','4','6','7','8'
                          ],
    'Th17(fromTreg)':['0',],#CD127, 或Th22
    'Th2(fromTreg)':['5',],#CD127
    'Doublet|Lowquality':['9',#CSF3R
                         '10'],
            }

#Single-cell atlas of healthy human blood unveils agerelated loss of NKG2C+GZMB–CD8+ memory T cells and accumulation of type 2 memory T cells

#naïve (CD45RApos), memory (CD45RAnegCD73neg), TR1‐like (CD73pos), and activated (HLA‐DRposCD39pos). 

In [109]:
check_dict_duplicates(cell_dict)

'无重复'

In [110]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [111]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    472807
Name: count, dtype: int64

In [112]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Treg memory CD4+ T    255480
Treg Naïve CD4+ T     113501
Th17(fromTreg)         76515
Th2(fromTreg)          22200
Doublet|Lowquality      5111
Name: count, dtype: int64

In [113]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [114]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Treg memory CD4+ T    255480
Treg Naïve CD4+ T     113501
Th17(fromTreg)         76515
Th2(fromTreg)          22200
Name: count, dtype: int64

In [115]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 2. 定 CD8 Treg Cell Refine

In [12]:
celltype="TregCD8"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [13]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','receptor_type','Celltype_L4_L5_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [14]:
groupby = "L4_leiden_TOTALVI_0.3"

In [312]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [314]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False,layer='denoised_protein')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='IL2RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='MKI67',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
sc.pl.dotplot(adata, ['ISG15','CD160','KLRD1','IL2RA','CTLA4','FOXP3','FCRL3','CCR10','RORC','KLRB1','HLA-DRB1',
                     'HAVCR2','LAG3','IKZF2','IKZF3','TIGIT','ENTPD3','IL7R','TNFRSF18'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CD40LG','CCR4','CCR6','CXCR5','KLRB1','GZMK','CCL5','BTBD9','CXCR3',
                      'GATA3','CCR10','LIMS1','RORC','CD27','CCR1','CCR2','PRDM1'],
              standard_scale='var',groupby=groupby)

In [15]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
0    2827
2    2457
1     783
3     197
Name: count, dtype: int64

In [16]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#IRF4- TGFB1+ CXCR4lo 
cell_dict = {
    'Treg CD8+ T':['0',],#
    "Doublet|Lowquality":['1',#似Vd1,但没有标记 CD4-CD8-
                          '2',#low UMI 加入后导致下游报错？CXCR5+ Th
                          '3',],#CD16
            }
#Single-cell atlas of healthy human blood unveils agerelated loss of NKG2C+GZMB–CD8+ memory T cells and accumulation of type 2 memory T cells

#naïve (CD45RApos), memory (CD45RAnegCD73neg), TR1‐like (CD73pos), and activated (HLA‐DRposCD39pos). 

In [17]:
check_dict_duplicates(cell_dict)

'无重复'

In [18]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [19]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    6264
Name: count, dtype: int64

In [20]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Doublet|Lowquality    3437
Treg CD8+ T           2827
Name: count, dtype: int64

In [21]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [22]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Treg CD8+ T    2827
Name: count, dtype: int64

In [23]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 3. 定 Helper memory CD4+ T Cell Refine

In [198]:
celltype="CD4_helper_memory"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [199]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [200]:
groupby = "L4_leiden_TOTALVI_1.5"

In [ ]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

In [ ]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [ ]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L3_L4_Refine',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=2,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data')

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='var',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_1']=="11",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [134]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1.5
0     293490
9     204413
1     189754
10    188602
4     188101
2     187552
6     171897
3     159214
13    158289
11    151003
8     123692
7     116417
14    111961
12     88665
18       150
5         52
17        40
25        22
35        20
34        20
16        19
24        16
22        14
36        10
30        10
29         9
26         8
20         8
32         8
31         7
27         7
21         6
28         6
33         6
19         5
23         3
15         2
Name: count, dtype: int64

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='var',groupby=groupby)

In [185]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {'Th22':['12','7'],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             'Th2':['13'],#PTGDR2=CRTH2 GATA3++ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             'Th1':['11',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+   CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'Th17':['4','6'],#CCR6=CD196++ RORC+ CCR4- KLRB1+/- GZMK- CCL5-    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['14','9'],#EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             'Tfh/Tcm':['0','1','2','3','5','8','10','18'],#CD45RA-。高表达CD27,用于形成记忆，不表达CCR7等,表达SELL
             'Doublet|Lowquality':['15','16','17','19','20','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36',#lncRNA expression and few cells
                         ],#
            }

In [186]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [187]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    2333498
Name: count, dtype: int64

In [188]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Tfh/Tcm               1142506
Th17                   359998
Th1/Th17               316374
Th22                   205082
Th2                    158289
Th1                    151003
Doublet|Lowquality        246
Name: count, dtype: int64

In [189]:
adata = adata[adata.obs['L4_leiden_TOTALVI_0.5'] != "5",:]#CD8
adata = adata[adata.obs['L4_leiden_TOTALVI_0.5'] != "4",:]#Naive Treg混合

In [190]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [191]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Tfh/Tcm     1126130
Th17         358773
Th1/Th17     316216
Th22         205017
Th2          158241
Th1          150899
Name: count, dtype: int64

In [ ]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 4. 定 Cytotoxic CD4+ T Cell Refine

In [193]:
celltype="CytotoxicCD4"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [194]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [195]:
groupby = "L4_leiden_TOTALVI_1.5"

In [500]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [507]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='GZMB',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='clr')
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adt, color='CD56',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='clr')
plt.show()

In [ ]:
adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[0]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','HAVCR2','LAG3',
                      'CCL5','CTLA4','CD40LG','CRTAM'],
              standard_scale='var',groupby=groupby)

In [73]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1.5
13    82559
8     82138
11    77285
4     76272
2     72874
1     72173
9     62024
5     60407
6     59240
0     57815
10    30807
18    27780
14    18564
7     17139
3     15436
19    14391
15     4831
12     1680
17     1484
16      669
20       23
21        9
22        6
Name: count, dtype: int64

In [74]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)


cell_dict = {'Terminal effector CD4+ T':['0','1','2','3','4','5','6','7','8','9','11','12','13','16','17',
                                        '15',],#Tex
             
             'Temra CD4+ T':['18'],#
             'Terminal effector HLA-DRhi CD4+ T':['10'],#

             'Th1':['19'],#
             'Doublet|Lowquality':['21','22','20',
                                   '0',#ADT CD183 CD186
                                   #HBB
                                   '14'#同时存在Naive和GZMB 
                                  ],#
            }

In [75]:
check_dict_duplicates(cell_dict)

'有重复'

In [76]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [77]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    835606
Name: count, dtype: int64

In [78]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Terminal effector CD4+ T             686211
Doublet|Lowquality                    76417
Terminal effector HLA-DRhi CD4+ T     30807
Temra CD4+ T                          27780
Th1                                   14391
Name: count, dtype: int64

In [79]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [80]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Terminal effector CD4+ T             686211
Terminal effector HLA-DRhi CD4+ T     30807
Temra CD4+ T                          27780
Th1                                   14391
Name: count, dtype: int64

In [81]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 5. 定 终 Tcm CD8+ T Cell Refine

In [37]:
celltype="CD8Tcm"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [38]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [39]:
groupby = "L4_leiden_TOTALVI_0.8"

In [111]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [113]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD8A',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'CXCR3',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CCL5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.violin(adt,'CD45RA',groupby=groupby,show=False, ax=axs[1,3],size=0,layer='clr')
plt.show()

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT',
                      'CCL5','CTLA4','CD40LG','CCR9','TRDV1','MYB','NUCB2','FXYD2'],
              standard_scale='var',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy'] = adata.obs['L4_leiden_TOTALVI_1.5'] == adata.obs['L4_leiden_TOTALVI_1.5'].cat.categories[13]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [40]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
0    41563
4    32040
2    30927
3    19651
1    10898
6       29
5       24
Name: count, dtype: int64

In [41]:
#naive (CD45RA+ CD45RO- CD62L+ CCR7+)
#central memory CD45RO and high levels of TCF7 and LEF1 transcripts) CD27+ CD45RA− 
#effector memory T (Tem) CD8+ subpopulations that expressed either granzyme K or granzyme B   CD27− CD45RA−

cell_dict = {'CCR4- CD8+ Tcm':['4',],#CD44 CCR4- CCL5+
             #CD45RA- KLRG1- TBX21- TCF7hi LEF1hi GATA3+
             #NELL2 #immunity Tcm CCR4+  TBX21- PRDM1=BLIMP1- EOMES- KLRG1-
             #CCR4+ GATA3+确认Tcm  CRIP2 GPR183
             #immunity Tcm   PRDM1=BLIMP1+ EOMES- KLRG1-
             #分为Tcm CCR4- CCL5+ GZMKlo；Tcm CCR4+ CCL5- GZMK-； 
             'CCR4+ CD8+ Tcm':['0','1','2','3',],#
             'Doublet|Lowquality':['6','5'],
            }

In [42]:
check_dict_duplicates(cell_dict)

'无重复'

In [43]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [44]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    135132
Name: count, dtype: int64

In [45]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
CCR4+ CD8+ Tcm        103039
CCR4- CD8+ Tcm         32040
Doublet|Lowquality        53
Name: count, dtype: int64

In [46]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [47]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
CCR4+ CD8+ Tcm    103039
CCR4- CD8+ Tcm     32040
Name: count, dtype: int64

In [48]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2',groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0864_M_Rep2_TATCAGCA_GTGTTCTA_AGTCACTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Tcm,CCR4+ CD8+ Tcm,CD8Tcm c0,NaN,NaN,3.049140,9.072036
D0428_Rep2_ACCTCCAA_CCATCCTC_CAGCGTTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Tcm,CCR4+ CD8+ Tcm,CD8Tcm c0,TCR,NaN,2.568181,10.681501
D0428_Rep2_GAGCTGAA_TAGGATGA_CACTTCGA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Tcm,CCR4+ CD8+ Tcm,CD8Tcm c0,TCR,NaN,1.281535,12.067717
D0428_Rep2_AGATGTAC_CTAAGGTC_ATCATTCC,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Tcm,CCR4+ CD8+ Tcm,CD8Tcm c0,NaN,NaN,1.328368,12.354582
D0428_Rep2_TAGGATGA_TGGAACAA_TTCACGCA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Tcm,CCR4+ CD8+ Tcm,CD8Tcm c2,TCR,NaN,0.983417,8.106975


# 6. 定 终 Tem CD8+ T Cell Refine

In [21]:
celltype="TemCD8"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [22]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','VJ_1_v_call'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           legend_loc='on data')

In [23]:
groupby = "L4_leiden_TOTALVI_1"

In [ ]:
adata.obs[groupby].value_counts()

In [365]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
cell_index = celltypist.samples.downsample_adata(adata,mode = 'each', n_cells = 5000,
                                                 by = groupby,
                                                 return_index = True,random_state=0)

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [57]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='GZMB',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'KLRG1',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'TRAC',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'TRDC',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adt,'CD27',groupby=groupby,show=False, ax=axs[1,3],size=0,layer='clr')
plt.show()

In [ ]:
sc.pl.dotplot(adata, ['KLRF1','CD27','CD28','KLRC2','CXCR3','KLRK1','ZNF683',
                      
                      'CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT',
                      'CCL5','CTLA4','CD40LG','CCR9','CCR5','HLA-DRA','HLA-DRB1','CD74'],
              standard_scale='obs',groupby=groupby)

In [24]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1
3     873202
1     747021
0     666591
4     610622
5     600238
2     233047
6       2892
34        37
12        33
36        25
33        22
19        20
25        14
30        12
46        11
13        11
15        10
26        10
28         9
21         9
27         9
10         8
32         8
29         8
24         8
50         8
16         8
18         8
23         8
35         7
49         6
48         6
43         6
8          6
31         6
38         6
40         6
41         5
47         5
14         5
9          5
20         5
7          4
11         4
42         4
22         4
37         4
39         4
51         4
44         4
45         3
17         2
Name: count, dtype: int64

In [25]:
#naive (CD45RA+ CD45RO- CD62L+ CCR7+)
#central memory CD45RO and high levels of TCF7 and LEF1 transcripts) CD27+ CD45RA− 
#effector memory T (Tem) CD8+ subpopulations that expressed either granzyme K or granzyme B   CD27− CD45RA−
#CD45RA+ CCL4 GZMK- Temra

cell_dict = {'Doublet|Lowquality':[''],#Too few cell with lncRNA
             'GZMK+ CD8+ Tem':['0',],#GZMK+ CD27+ TCF7++
             'GZMB+ CD8+ Tem':['2','3',],# GZMB+ CD27- ZNF683
             'HLA-DRhi CD8+ Tem':['1',],#GZMB+ CD27+ ZNF683- CCR5+ HLA-DRA HLA-DRB1 CD74
             'CD8+ Temra':['4','5','6'],#GZMB+ KLRB1+ CD45RA+ KLRF1+ 
            }

In [26]:
check_dict_duplicates(cell_dict)

'无重复'

In [27]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [28]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    3733613
True         407
Name: count, dtype: int64

In [29]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
CD8+ Temra           1213752
GZMB+ CD8+ Tem       1106249
HLA-DRhi CD8+ Tem     747021
GZMK+ CD8+ Tem        666591
Name: count, dtype: int64

In [30]:
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_0.3']=="2",'Celltype_L4_L5_Refine_R2'] = "vNKT"#type 2 variant NKT

In [31]:
#去除没有注释的细胞 lncRNA+ too few cells
adata = adata[(~adata.obs['Celltype_L4_L5_Refine_R2'].isna()),:]

In [32]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
CD8+ Temra           1134447
GZMB+ CD8+ Tem       1105417
HLA-DRhi CD8+ Tem     747000
GZMK+ CD8+ Tem        666573
vNKT                   80176
Name: count, dtype: int64

In [36]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2',groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0589_Rep1_AACTCACC_AGATCGCA_CAAGGAGC,CD4+ T,Treg,Tem CD8+ T,Tem CD8+ T,Tem GZMB+ CD8+ T,GZMK+ CD8+ Tem,TemCD8 c0,TCR,NaN,4.617856,8.212212
D0589_Rep1_CCTCCTGA_ACATTGGC_TAGGATGA,CD4+ T,Treg,Tem CD8+ T,Tem CD8+ T,Tem GZMK+ CD8+ T,GZMK+ CD8+ Tem,TemCD8 c0,TCR,NaN,5.656895,10.175475
D0864_M_Rep2_GGTGCGAA_AGCACCTC_AGCACCTC,CD4+ T,Treg,Tem CD8+ T,Tem CD8+ T,Tem GZMK+ CD8+ T,HLA-DRhi CD8+ Tem,TemCD8 c1,TCR,NaN,4.005968,11.216032
D0878_Rep2_GAACAGGC_GTACGCAA_CTAAGGTC,CD4+ T,Treg,Tem CD8+ T,Tem CD8+ T,Tem GZMK+ CD8+ T,HLA-DRhi CD8+ Tem,TemCD8 c1,TCR,NaN,3.021657,8.540588
D0878_Rep2_TTCACGCA_AGAGTCAA_GCGAGTAA,CD4+ T,Treg,Tem CD8+ T,Tem CD8+ T,Tem HLA-DRhi CD8+ T,HLA-DRhi CD8+ Tem,TemCD8 c1,TCR,NaN,4.395370,9.915993


# 7. 定 MAIT Cell Refine

In [60]:
celltype="MAIT"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [61]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [62]:
adata.obs['Celltype_L4_L5_Refine'].value_counts()

Celltype_L4_L5_Refine
MAIT    322468
Name: count, dtype: int64

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [64]:
groupby = "L4_leiden_TOTALVI_0.3"

In [34]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [36]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRAV1-2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
#sc.pl.violin(adt,'CD45RA',groupby=groupby,show=False, ax=axs[0,3],size=0.5)
#TNFAIP3/CXCR4/DDIT4/FOSL2/IRS2/BTG1/ZFP36/PNRC1/CDKN1B/PLIN2/BHLHE40/PLAUR/ETS1/DUSP1
#SELL/TXNIP/IFITM2/B2M/HLA-C

sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'KLRB1',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'TRAV1-2',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'SLC4A10',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()

In [39]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
1    284925
0     37543
Name: count, dtype: int64

In [ ]:
sc.pl.dotplot(adata, ['ISG15','JUN','FOS','IL32','TRAV1-2','SLC4A10','KLRB1','S100A4','LTB','GNLY','KLRD1','TRDV2','TRGV9'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
#CD185 (CXCR5) CD183 (CXCR3) CD278 (ICOS) CD279 (PD1)
sc.pl.dotplot(adt, ['CD45RA','CD197','CD62L','CD161','CD279','CD185','CD183','CD278','CD279','CD27','CD25'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT',
                      'CCL5','CTLA4','CD40LG','CCR9','FCRL6','FCRL3'],
              standard_scale='obs',groupby=groupby)

In [67]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
1    284925
0     37543
Name: count, dtype: int64

In [68]:
cell_dict={
    'MAIT':['1'],#
    'Doublet|Lowquality':['0'],#CD8, TRAV1-2 too low, No SCL4A10
}

#存在CD56- MAIT和CD56+ MAIT(8) https://www.nature.com/articles/s41590-023-01575-1#Abs1

In [69]:
check_dict_duplicates(cell_dict)

'无重复'

In [70]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [71]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    322468
Name: count, dtype: int64

In [72]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
MAIT                  284925
Doublet|Lowquality     37543
Name: count, dtype: int64

In [73]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [74]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
MAIT    284925
Name: count, dtype: int64

In [75]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 8. 定 Vd1 Refine

In [243]:
celltype="Vd1"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [244]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [245]:
groupby = "L4_leiden_TOTALVI_0.5"

In [386]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [388]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRF1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.umap(adata, color='TRDV2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.violin(adata,'TRGV9',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adt,'CD27',groupby=groupby,show=False, ax=axs[1,3],size=0.5,layer='clr')
plt.show()

In [ ]:
sc.pl.dotplot(adt, ['CD27','CD45RA','CD197','CD62L','CD127','CD161','CD16','CD56'],
              groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['TRDV2','TRGV9','TRDV1','KLRF1','TIGIT','FCGR3A','CD27','SELL','CCR7','IL7R','KLRF1','NCR1','CMC1',
                      'CCL5','TCF7','LEF1','SLC4A10','GZMK','KLRD1','KLRB1','KLRK1','BTN3A1','FCGR3A'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9','FCRL3',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='var',groupby=groupby)

In [246]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
2    18118
0    16580
3    10313
1     6430
Name: count, dtype: int64

In [247]:
cell_dict={
    'Vδ1 SOX4+ T':['2',],#
    'Naïve Vδ1 T':['0',],#
    'Doublet|Lowquality':['1','3'],#CD8A CD8B
}


#gdT 在血液中多是CD45RA- CD27+/-

# #γδ T；https://www.nature.com/articles/s41467-018-04076-0；
# https://www.nature.com/articles/s41392-023-01653-8#Sec12 #CD8+ gdT=Vδ1+  CD56+γδT存在
# Vδ2+ T cells were predominantly CD27+CD45RA−: https://journals.aai.org/jimmunol/article/197/12/4584/109106/CD8-T-Cells-A-Novel-T-Cell-Subset-with-a-Potential

# Vδ1+ T  https://www.cell.com/cell-reports/fulltext/S2211-1247(22)00631-3

# γδT细胞根据TCR的γ（包括2/3/4/5/8/9）和δ（包括1/2/3/5）链的表达，γδT细胞主要分为三个亚群：Vδ1T细胞，Vδ2T细胞和Vδ3T细胞。
# Vδ1T细胞主要存在于粘膜上皮细胞中，Vδ2T细胞主要分布在外周血中，Vδ3T细胞主要分布肝和肠。
# 基于CD27和CD45RA的表达差异Vδ2T细胞又分为CD45RA + CD27 +（幼稚），CD45RA-CD27 +（中间，无记忆效应功能），
# CD45RA-CD27-（记忆效应）和CD45RA + CD27-（终末分化）四群。
# 另外，γδT细胞也可分为多个功能亚群：产IFN-γ，产IL-17AγδT细胞和抗原呈递γδT细胞。

In [248]:
check_dict_duplicates(cell_dict)

'无重复'

In [249]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [250]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    51441
Name: count, dtype: int64

In [251]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Vδ1 SOX4+ T           18118
Doublet|Lowquality    16743
Naïve Vδ1 T           16580
Name: count, dtype: int64

In [252]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [253]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Vδ1 SOX4+ T    18118
Naïve Vδ1 T    16580
Name: count, dtype: int64

In [254]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 9. 定 Vd2 Refine

In [255]:
celltype="Vd2"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [256]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [257]:
groupby = "L4_leiden_TOTALVI_0.3"

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="3",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [409]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [411]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRF1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.umap(adata, color='TRDV2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CD8A',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adt,'CD27',groupby=groupby,show=False, ax=axs[1,3],size=0.5,layer='clr')
plt.show()

In [ ]:
sc.pl.dotplot(adt, ['CD27','CD45RA','CD197','CD62L','CD127','CD161','CD16','CD56'],
              groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['TRDV2','TRGV9','TRDV1','KLRF1','TIGIT','FCGR3A','CD27','SELL','CCR7','IL7R','KLRF1','NCR1','CMC1',
                      'CCL5','TCF7','LEF1','SLC4A10','GZMK','KLRD1','KLRB1','KLRK1','BTN3A1','FCGR3A'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9','FCRL3',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='var',groupby=groupby)

In [258]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
1    398020
4    312281
2    187450
0     39664
3      4867
5      2787
Name: count, dtype: int64

In [259]:
adata.obs.groupby(['L4_leiden_TOTALVI_0.3','L4_leiden_TOTALVI_1.5']).size()

L4_leiden_TOTALVI_0.3  L4_leiden_TOTALVI_1.5
0                      0                        29651
                       1                           78
                       2                           10
                       3                          102
                       4                           66
                                                ...  
5                      17                           0
                       18                           0
                       19                           0
                       20                           0
                       21                        2787
Length: 132, dtype: int64

In [260]:
cell_dict={
    'Vδ1 KLRF1- T':['3',],
    'Vδ2 T':['1','2','4'],
    'Doublet|Lowquality':['5',#
                         '0',],#CD16 CSF3R
}
#gdT 在血液中多是CD45RA- CD27+/-

# #γδ T；https://www.nature.com/articles/s41467-018-04076-0；
# https://www.nature.com/articles/s41392-023-01653-8#Sec12 #CD8+ gdT=Vδ1+  CD56+γδT存在
# Vδ2+ T cells were predominantly CD27+CD45RA−: https://journals.aai.org/jimmunol/article/197/12/4584/109106/CD8-T-Cells-A-Novel-T-Cell-Subset-with-a-Potential

# Vδ1+ T  https://www.cell.com/cell-reports/fulltext/S2211-1247(22)00631-3

# γδT细胞根据TCR的γ（包括2/3/4/5/8/9）和δ（包括1/2/3/5）链的表达，γδT细胞主要分为三个亚群：Vδ1T细胞，Vδ2T细胞和Vδ3T细胞。
# Vδ1T细胞主要存在于粘膜上皮细胞中，Vδ2T细胞主要分布在外周血中，Vδ3T细胞主要分布肝和肠。
# 基于CD27和CD45RA的表达差异Vδ2T细胞又分为CD45RA + CD27 +（幼稚），CD45RA-CD27 +（中间，无记忆效应功能），
# CD45RA-CD27-（记忆效应）和CD45RA + CD27-（终末分化）四群。
# 另外，γδT细胞也可分为多个功能亚群：产IFN-γ，产IL-17AγδT细胞和抗原呈递γδT细胞。

In [261]:
check_dict_duplicates(cell_dict)

'无重复'

In [262]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [263]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    945069
Name: count, dtype: int64

In [264]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Vδ2 T                 897751
Doublet|Lowquality     42451
Vδ1 KLRF1- T            4867
Name: count, dtype: int64

In [265]:
adata.obs['L4_leiden_TOTALVI_1.5'].value_counts()

L4_leiden_TOTALVI_1.5
7     87515
12    84272
8     77331
15    76210
13    72429
14    70135
2     70049
1     67643
10    61644
0     56307
9     53263
4     44327
3     42007
6     36489
5     13262
11    11380
17    11227
19     3608
21     2801
16     1384
18     1032
20      754
Name: count, dtype: int64

In [266]:
adata = adata[adata.obs['L4_leiden_TOTALVI_1.5'] != "18",:]#HBB

In [267]:
adata = adata[adata.obs['L4_leiden_TOTALVI_1.5'] != "20",:]#JCHAIN Vd1 MKI67

In [268]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R2'] != "Doublet|Lowquality",:]

In [269]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Vδ2 T           895965
Vδ1 KLRF1- T      4867
Name: count, dtype: int64

In [270]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R5_refine_indices_{celltype}.csv")

# 10. 定 ProliferativeT Cell Refine

In [56]:
celltype="ProliferativeT"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [57]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           legend_loc='on data')

In [58]:
groupby = "L4_leiden_TOTALVI_1.5"

In [442]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [444]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False,layer='denoised_protein')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='IL2RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='STMN1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
sc.pl.dotplot(adata, ['ISG15','CD160','KLRD1','IL2RA','CTLA4','FOXP3','FCRL3','CCR10','RORC','KLRB1','HLA-DRB1',
                     'HAVCR2'],
              standard_scale='var',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['MKI67','STMN1','PCNA','CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [59]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1.5
14    3220
10    2966
5     2746
13    2739
9     2637
4     2269
11    2199
2     1751
18    1677
15    1484
7     1308
8     1207
17    1090
19     957
1      877
0      754
12     693
3      688
6      503
16     316
21     279
20      91
Name: count, dtype: int64

In [60]:
#增殖CD38 MKI67 STMN1

cell_dict = {'Proliferative Treg':['9','10','19'],
            'Proliferative memory CD8+ T':['0','4','5','6','12','18',],
            'Proliferative Temra CD8+ T':['8','14','17',],
            'Proliferative help memory CD4+ T':['1','2','7','11','13','15','20'],
            'Proliferative Cytotoxic CD4+ T':['16'],
    'Proliferative Dn T':['3'],#
    'Proliferative γδT T':['21'],#
            }

In [61]:
check_dict_duplicates(cell_dict)

'无重复'

In [62]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R2'] = i

In [63]:
(adata.obs['Celltype_L4_L5_Refine_R2'].isna()).value_counts()

Celltype_L4_L5_Refine_R2
False    32451
Name: count, dtype: int64

In [64]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Proliferative help memory CD4+ T    10449
Proliferative memory CD8+ T          8642
Proliferative Treg                   6560
Proliferative Temra CD8+ T           5517
Proliferative Dn T                    688
Proliferative Cytotoxic CD4+ T        316
Proliferative γδT T                   279
Name: count, dtype: int64

In [65]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0]  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1]  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")

# 合并所有数据

## PBMC et. al.

In [66]:
try:
    del indices_dict,combined_indices,Finnal_indices_dict,combined_Finnal_indices_dict,Processing_combined_indices
except:
    print("It is None!")

It is None!


## 终版

In [67]:
celltypes=['AtypicalB',
           'Basophil','CEACAM8_Pos_Neutrophil','CEACAM8_Neg_Neutrophil',
           'CD8Tcm',
           'DC','DnT',
           'HSPC',
           'iNKT','Mast','Memory_B',
           'Monocyte','NaiveB','NaiveCD4','NaiveCD8','NK', 
           'Non_NK_ILC','Plasma','Platelet',
           'ProliferativeT','TemCD8',
]

In [68]:
Finnal_indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{finnal_path}{i}/Finnal_indices_{i}.csv',index_col=0)
    Finnal_indices_dict.append(indices)

100%|██████████| 21/21 [00:52<00:00,  2.52s/it]


In [69]:
combined_Finnal_indices_dict = pd.concat(Finnal_indices_dict)

In [70]:
combined_Finnal_indices_dict['Celltype_L4_L5_Refine_R2'].value_counts()
#本轮获得的终版细胞数1742676

Celltype_L4_L5_Refine_R2
Temra CD8+ T                        1134447
Tem GZMB+ CD8+ T                    1105417
Tem HLA-DRhi CD8+ T                  747000
Tem GZMK+ CD8+ T                     666573
Tcm CCR4+ CCL5- CD8+ T               103039
vNKT                                  80176
Tcm CCR4- CCL5+ CD8+ T                32040
Proliferative help memory CD4+ T      10449
Proliferative memory CD8+ T            8642
Proliferative Treg                     6560
Proliferative Temra CD8+ T             5517
Proliferative Dn T                      688
Proliferative Cytotoxic CD4+ T          316
Proliferative γδT T                     279
Name: count, dtype: int64

## 过程

In [71]:
%%bash
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R5/ | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R5/*/R5_refine* | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/ | wc -l

10
8
21


In [72]:
celltypes=['TregCD8',
           'TregCD4',
           'CD4_helper_memory',
           'CytotoxicCD4',
           'MAIT',
           'Vd1',
           'Vd2',
          ]

In [73]:
indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{obj_path}{i}/R5_refine_indices_{i}.csv',index_col=0)
    indices_dict.append(indices)

100%|██████████| 7/7 [00:04<00:00,  1.46it/s]


In [74]:
Processing_combined_indices = pd.concat(indices_dict)

In [75]:
combined_indices = pd.concat([Processing_combined_indices,combined_Finnal_indices_dict])

In [76]:
combined_indices['Celltype_L1_L2'].isna().value_counts()

Celltype_L1_L2
False    52083406
Name: count, dtype: int64

In [77]:
combined_indices['Celltype_L1_L2_Refine'].isna().value_counts()

Celltype_L1_L2_Refine
False    52083406
Name: count, dtype: int64

In [78]:
combined_indices['Celltype_L2_L3_Refine'].isna().value_counts()

Celltype_L2_L3_Refine
False    52083406
Name: count, dtype: int64

In [79]:
combined_indices['Celltype_L3_L4_Refine'].isna().value_counts()

Celltype_L3_L4_Refine
True     37417894
False    14665512
Name: count, dtype: int64

In [80]:
combined_indices['Celltype_L4_L5_Refine_R2'].isna().value_counts()

Celltype_L4_L5_Refine_R2
True     43416820
False     8666586
Name: count, dtype: int64

In [81]:
combined_indices['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Temra CD8+ T                         1134447
Tfh/Tcm                              1126130
Tem GZMB+ CD8+ T                     1105417
Vδ2 T                                 895965
Tem HLA-DRhi CD8+ T                   747000
Terminal effector CD4+ T              686211
Tem GZMK+ CD8+ T                      666573
Th17                                  358773
Th1/Th17                              316216
MAIT                                  284925
Treg memory CD4+ T                    255480
Th22                                  205017
Th1                                   165290
Th2                                   158241
Treg Naïve CD4+ T                     113501
Tcm CCR4+ CCL5- CD8+ T                103039
vNKT                                   80176
Th17(fromTreg)                         76515
Tcm CCR4- CCL5+ CD8+ T                 32040
Terminal effector HLA-DRhi CD4+ T      30807
Temra CD4+ T                           27780
Th2(fromTreg)                 

## save

In [82]:
def save_to_indices(celltype,celltype_to_file):
    L2_refine_path_R6 = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R6'
    indices=combined_indices['Celltype_L4_L5_Refine_R2'].isin(celltype)
    indices_celltype=combined_indices[indices]
    print(f"{indices_celltype['Celltype_L4_L5_Refine_R2'].value_counts()}")
    os.makedirs(f'{L2_refine_path_R6}/{celltype_to_file}', exist_ok=True)
    indices_celltype.to_csv(f"{L2_refine_path_R6}/{celltype_to_file}/R6_indices_{celltype_to_file}.csv")

In [83]:
Processing_combined_indices['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Tfh/Tcm                              1126130
Vδ2 T                                 895965
Terminal effector CD4+ T              686211
Th17                                  358773
Th1/Th17                              316216
MAIT                                  284925
Treg memory CD4+ T                    255480
Th22                                  205017
Th1                                   165290
Th2                                   158241
Treg Naïve CD4+ T                     113501
Th17(fromTreg)                         76515
Terminal effector HLA-DRhi CD4+ T      30807
Temra CD4+ T                           27780
Th2(fromTreg)                          22200
Vδ1 SOX4+ T                            18118
Naïve Vδ1 T                            16580
Vδ1 KLRF1- T                            4867
Treg CD8+ T                             2827
Name: count, dtype: int64

In [639]:
save_to_indices(celltype=['MAIT',],celltype_to_file='MAIT')

Celltype_L4_L5_Refine_R2
MAIT    284925
Name: count, dtype: int64


In [289]:
save_to_indices(celltype=['Vδ2 T',],celltype_to_file='Vd2')

Celltype_L4_L5_Refine_R2
Vδ2 T    895965
Name: count, dtype: int64


In [290]:
save_to_indices(celltype=['Vδ1 SOX4+ T','Naïve Vδ1 T','Vδ1 KLRF1- T'],celltype_to_file='Vd1')

Celltype_L4_L5_Refine_R2
Vδ1 SOX4+ T     18118
Naïve Vδ1 T     16580
Vδ1 KLRF1- T     4867
Name: count, dtype: int64


In [641]:
save_to_indices(celltype=['Treg CD8+ T'],celltype_to_file='TregCD8')
save_to_indices(celltype=['Treg memory CD4+ T','Treg Naïve CD4+ T',],celltype_to_file='TregCD4')

Celltype_L4_L5_Refine_R2
Treg CD8+ T    2827
Name: count, dtype: int64
Celltype_L4_L5_Refine_R2
Treg memory CD4+ T    255480
Treg Naïve CD4+ T     113501
Name: count, dtype: int64


In [155]:
save_to_indices(celltype=['Th1/Th17','Th22','Th17','Th1','Th2','Th17(fromTreg)','Th2(fromTreg)'],celltype_to_file='Th1_17_2_22')

Celltype_L4_L5_Refine_R2
Th17              358773
Th1/Th17          316216
Th22              205017
Th1               165290
Th2               158241
Th17(fromTreg)     76515
Th2(fromTreg)      22200
Name: count, dtype: int64


In [153]:
save_to_indices(celltype=['Tfh/Tcm',],celltype_to_file='Tfh_Tcm')

Celltype_L4_L5_Refine_R2
Tfh/Tcm    1126130
Name: count, dtype: int64


In [100]:
save_to_indices(celltype=['Temra CD4+ T','Terminal effector HLA-DRhi CD4+ T','Terminal effector CD4+ T',],celltype_to_file='CytotoxicCD4')

Celltype_L4_L5_Refine_R2
Terminal effector CD4+ T             686211
Terminal effector HLA-DRhi CD4+ T     30807
Temra CD4+ T                          27780
Name: count, dtype: int64
